In [1]:
from pathlib import Path
from contextlib import nullcontext
import gc
import json
import re
import tarfile
import time

import numpy as np
import pandas as pd
import requests
import sacrebleu
import torch
from peft import PeftModel
from tqdm.auto import tqdm
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
CACHE_DIR = PROJECT_ROOT / 'data/external/trusted_benchmarks'
RESULT_DIR = PROJECT_ROOT / 'results/trusted_benchmark_eval'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
RESULT_DIR.mkdir(parents=True, exist_ok=True)
BASE_MODEL = 'facebook/m2m100_418M'
ADAPTER_PATH = PROJECT_ROOT / 'models/lora/m2m100_en_uz_hplt_smoke/final_adapter'
MAX_PAIRS_PER_DIRECTION = 200
MAX_SOURCE_LENGTH = 128
MAX_NEW_TOKENS = 128
SEED = 42
RUN_BASELINE = True
RUN_ADAPTER = True
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE = torch.bfloat16 if DEVICE == 'cuda' and torch.cuda.is_bf16_supported() else (torch.float16 if DEVICE == 'cuda' else torch.float32)
assert ADAPTER_PATH.exists(), f'Adapter not found: {ADAPTER_PATH}'
print('Device:', DEVICE, 'dtype:', DTYPE)
print('Adapter:', ADAPTER_PATH)
print('Results:', RESULT_DIR)

D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda dtype: torch.bfloat16
Adapter: D:\dev\projects\fourlang_translation\models\lora\m2m100_en_uz_hplt_smoke\final_adapter
Results: D:\dev\projects\fourlang_translation\results\trusted_benchmark_eval


In [2]:
FLORES_URL = 'https://dl.fbaipublicfiles.com/nllb/flores200_dataset.tar.gz'
FLORES_ARCHIVE = CACHE_DIR / 'flores200_dataset.tar.gz'
NTREX_URLS = {
    'en': 'https://huggingface.co/datasets/davidstap/NTREX/resolve/refs%2Fconvert%2Fparquet/eng_Latn/test/0000.parquet',
    'uz': 'https://huggingface.co/datasets/davidstap/NTREX/resolve/refs%2Fconvert%2Fparquet/uzb_Latn/test/0000.parquet',
}

def download_file(url, path):
    if path.exists() and path.stat().st_size > 0:
        return path
    temp = path.with_suffix(path.suffix + '.part')
    with requests.get(url, stream=True, timeout=120) as response:
        response.raise_for_status()
        total = int(response.headers.get('content-length', 0))
        with open(temp, 'wb') as handle, tqdm(total=total, unit='B', unit_scale=True, desc=path.name) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    handle.write(chunk)
                    bar.update(len(chunk))
    temp.replace(path)
    return path

download_file(FLORES_URL, FLORES_ARCHIVE)
for lang, url in NTREX_URLS.items():
    download_file(url, CACHE_DIR / f'ntrex_{lang}.parquet')
print('Benchmark source files are ready.')

Benchmark source files are ready.


In [3]:
def read_tar_lines(archive_path, member_name):
    normalized = member_name.lstrip('./')
    with tarfile.open(archive_path, 'r:gz') as archive:
        matches = [m for m in archive.getmembers() if m.name.lstrip('./') == normalized]
        if not matches:
            matches = [m for m in archive.getmembers() if m.name.lstrip('./').endswith(normalized)]
        if len(matches) != 1:
            raise KeyError(f'Expected one archive member ending with {normalized!r}, found {len(matches)}')
        member = matches[0]
        handle = archive.extractfile(member)
        assert handle is not None
        return handle.read().decode('utf-8').splitlines()

def bidirectional_rows(benchmark, ids, en_values, uz_values):
    rows = []
    for pair_id, en, uz in zip(ids, en_values, uz_values):
        en, uz = str(en).strip(), str(uz).strip()
        if not en or not uz:
            continue
        rows.append({'benchmark': benchmark, 'pair_id': str(pair_id), 'src_lang': 'en', 'tgt_lang': 'uz', 'source': en, 'reference': uz})
        rows.append({'benchmark': benchmark, 'pair_id': str(pair_id), 'src_lang': 'uz', 'tgt_lang': 'en', 'source': uz, 'reference': en})
    return rows

all_rows = []
for split in ['devtest']:
    en_lines = read_tar_lines(FLORES_ARCHIVE, f'flores200_dataset/{split}/eng_Latn.{split}')
    uz_lines = read_tar_lines(FLORES_ARCHIVE, f'flores200_dataset/{split}/uzn_Latn.{split}')
    assert len(en_lines) == len(uz_lines)
    all_rows.extend(bidirectional_rows(f'flores_{split}', range(1, len(en_lines) + 1), en_lines, uz_lines))

ntrex_en = pd.read_parquet(CACHE_DIR / 'ntrex_en.parquet')['text'].astype(str).tolist()
ntrex_uz = pd.read_parquet(CACHE_DIR / 'ntrex_uz.parquet')['text'].astype(str).tolist()
assert len(ntrex_en) == len(ntrex_uz)
all_rows.extend(bidirectional_rows('ntrex', range(1, len(ntrex_en) + 1), ntrex_en, ntrex_uz))

tatoeba_path = PROJECT_ROOT / 'data/clean/en_uz/tatoeba_en_uz_latin.csv'
tatoeba = pd.read_csv(tatoeba_path).dropna(subset=['en', 'uz']).drop_duplicates(['en', 'uz'])
tatoeba_ids = tatoeba['en_id'].astype(str) + '-' + tatoeba['uz_id'].astype(str)
all_rows.extend(bidirectional_rows('tatoeba_latin', tatoeba_ids, tatoeba['en'], tatoeba['uz']))

benchmark_df = pd.DataFrame(all_rows).drop_duplicates(['benchmark', 'pair_id', 'src_lang', 'tgt_lang'])
sampled = []
for (benchmark, src_lang, tgt_lang), group in benchmark_df.groupby(['benchmark', 'src_lang', 'tgt_lang'], sort=True):
    n = min(MAX_PAIRS_PER_DIRECTION, len(group))
    sampled.append(group.sample(n=n, random_state=SEED).sort_values('pair_id'))
benchmark_df = pd.concat(sampled, ignore_index=True)
benchmark_df.insert(0, 'eval_id', [f'eval_{i:06d}' for i in range(len(benchmark_df))])
benchmark_df.to_csv(RESULT_DIR / 'benchmark_pairs.csv', index=False, encoding='utf-8-sig')
display(benchmark_df.groupby(['benchmark', 'src_lang', 'tgt_lang']).size().rename('samples').reset_index())
display(benchmark_df.head())

,benchmark,src_lang,tgt_lang,samples
0,flores_devtest,en,uz,200
1,flores_devtest,uz,en,200
2,ntrex,en,uz,200
3,ntrex,uz,en,200
4,tatoeba_latin,en,uz,200
5,tatoeba_latin,uz,en,200


,eval_id,benchmark,pair_id,src_lang,tgt_lang,source,reference
0,eval_000000,flores_devtest,1008,en,uz,"As the areas are sparsely populated, and light...","Hududlarda aholi siyrakligi tufayli, yorug'lik..."
1,eval_000001,flores_devtest,1011,en,uz,"Workplace harmony is crucial, emphasizing grou...",Alohida shaxslarning yutuqlarini maqtashdan ko...
2,eval_000002,flores_devtest,102,en,uz,"However, the percentage of XDR-TB in the entir...",Ammo sil kasalligi bilan xastalangan odamlarni...
3,eval_000003,flores_devtest,108,en,uz,A doctor who worked at Children's Hospital of ...,Pensilvaniya shtati Pittsburg shahridagi bolal...
4,eval_000004,flores_devtest,11,en,uz,While one experimental vaccine appears able to...,Birgina tajriba vaksinasi Eboladan o'lish xavf...


In [4]:
tokenizer = M2M100Tokenizer.from_pretrained(BASE_MODEL)

def load_eval_model(adapter_path=None):
    kwargs = {'torch_dtype': DTYPE, 'low_cpu_mem_usage': True}
    base = M2M100ForConditionalGeneration.from_pretrained(BASE_MODEL, **kwargs)
    if adapter_path is not None:
        base = PeftModel.from_pretrained(base, adapter_path)
    base.to(DEVICE)
    base.eval()
    base.config.use_cache = True
    return base

def synchronize():
    if DEVICE == 'cuda':
        torch.cuda.synchronize()

def repeated_phrase(text):
    return bool(re.search(r"\b([\w']+)\b(?:\s+\1\b){2,}", text.casefold(), flags=re.UNICODE))

@torch.inference_mode()
def translate_one(model, text, src_lang, tgt_lang):
    tokenizer.src_lang = src_lang
    started_total = time.perf_counter()
    inputs = tokenizer(text, return_tensors='pt', truncation=True, max_length=MAX_SOURCE_LENGTH).to(DEVICE)
    synchronize()
    started_generation = time.perf_counter()
    generated = model.generate(
        **inputs,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang),
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=1,
        repetition_penalty=1.15,
        no_repeat_ngram_size=3,
    )
    synchronize()
    generation_seconds = time.perf_counter() - started_generation
    prediction = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]
    total_seconds = time.perf_counter() - started_total
    return prediction, generation_seconds, total_seconds, int(generated.shape[-1])

print('Evaluation helpers are ready.')

Evaluation helpers are ready.


In [5]:
def evaluate_model(model_name, adapter_path=None):
    output_path = RESULT_DIR / f'predictions_{model_name}.csv'
    if output_path.exists():
        existing = pd.read_csv(output_path, keep_default_na=False)
    else:
        existing = pd.DataFrame()
    completed = set(existing['eval_id'].astype(str)) if len(existing) else set()
    pending = benchmark_df[~benchmark_df['eval_id'].isin(completed)]
    print(model_name, 'completed:', len(completed), 'pending:', len(pending))
    if not len(pending):
        return existing
    model = load_eval_model(adapter_path)
    _ = translate_one(model, 'Hello.', 'en', 'uz')
    new_rows = []
    for row in tqdm(pending.itertuples(index=False), total=len(pending), desc=model_name):
        prediction, generation_seconds, total_seconds, generated_tokens = translate_one(
            model, row.source, row.src_lang, row.tgt_lang
        )
        new_rows.append({
            **row._asdict(),
            'model_name': model_name,
            'prediction': prediction,
            'generation_seconds': generation_seconds,
            'total_seconds': total_seconds,
            'generated_tokens': generated_tokens,
            'has_repetition': repeated_phrase(prediction),
            'hit_max_tokens': generated_tokens >= MAX_NEW_TOKENS,
        })
        if len(new_rows) % 25 == 0:
            combined = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
            combined.to_csv(output_path, index=False, encoding='utf-8-sig')
    combined = pd.concat([existing, pd.DataFrame(new_rows)], ignore_index=True)
    combined = combined.drop_duplicates('eval_id', keep='last').sort_values('eval_id')
    combined.to_csv(output_path, index=False, encoding='utf-8-sig')
    del model
    gc.collect()
    if DEVICE == 'cuda':
        torch.cuda.empty_cache()
    return combined

baseline_predictions = evaluate_model('base', None) if RUN_BASELINE else None

base completed: 0 pending: 1200


D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
base:   2%|▏         | 25/1200 [00:05<04:31,  4.33it/s]D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
base:   4%|▍         | 50/1200 [00:10<04:09,  4.60it/s]D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based genera

In [6]:
adapter_predictions = evaluate_model('adapter', ADAPTER_PATH) if RUN_ADAPTER else None

adapter completed: 0 pending: 1200


D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
adapter:   2%|▏         | 25/1200 [00:12<10:39,  1.84it/s]D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based generation modes. You should set `num_beams>1` or unset `early_stopping`.
  warnings.warn(
adapter:   4%|▍         | 50/1200 [00:25<09:23,  2.04it/s]D:\dev\projects\fourlang_translation\.venv\Lib\site-packages\transformers\generation\configuration_utils.py:638: UserWarning: `num_beams` is set to 1. However, `early_stopping` is set to `True` -- this flag is only used in beam-based 

In [7]:
def truthy(series):
    if series.dtype == bool:
        return series
    return series.astype(str).str.lower().isin(['true', '1', 'yes'])

def compute_metrics(predictions):
    records = []
    for (benchmark, src_lang, tgt_lang), group in predictions.groupby(['benchmark', 'src_lang', 'tgt_lang']):
        preds = group['prediction'].astype(str).tolist()
        refs = group['reference'].astype(str).tolist()
        ref_lengths = group['reference'].astype(str).str.len().clip(lower=1)
        pred_lengths = group['prediction'].astype(str).str.len()
        records.append({
            'model_name': str(group['model_name'].iloc[0]),
            'benchmark': benchmark,
            'direction': f'{src_lang}-{tgt_lang}',
            'samples': len(group),
            'bleu': sacrebleu.corpus_bleu(preds, [refs]).score,
            'chrf2': sacrebleu.corpus_chrf(preds, [refs], word_order=2).score,
            'repetition_percent': float(truthy(group['has_repetition']).mean() * 100),
            'hit_max_tokens_percent': float(truthy(group['hit_max_tokens']).mean() * 100),
            'mean_length_ratio': float((pred_lengths / ref_lengths).mean()),
            'latency_mean_seconds': float(pd.to_numeric(group['total_seconds']).mean()),
            'latency_p95_seconds': float(pd.to_numeric(group['total_seconds']).quantile(.95)),
        })
    return pd.DataFrame(records)

metric_frames = []
for model_name in ['base', 'adapter']:
    path = RESULT_DIR / f'predictions_{model_name}.csv'
    if path.exists():
        metric_frames.append(compute_metrics(pd.read_csv(path, keep_default_na=False)))
metrics_df = pd.concat(metric_frames, ignore_index=True)
metrics_df.to_csv(RESULT_DIR / 'metrics_all.csv', index=False, encoding='utf-8-sig')
display(metrics_df.round(4))

if set(metrics_df['model_name']) >= {'base', 'adapter'}:
    key = ['benchmark', 'direction']
    base = metrics_df[metrics_df.model_name == 'base'].set_index(key)
    adapter = metrics_df[metrics_df.model_name == 'adapter'].set_index(key)
    comparison = adapter[['bleu', 'chrf2', 'repetition_percent', 'hit_max_tokens_percent', 'latency_p95_seconds']].subtract(
        base[['bleu', 'chrf2', 'repetition_percent', 'hit_max_tokens_percent', 'latency_p95_seconds']]
    ).add_prefix('delta_').reset_index()
    comparison.to_csv(RESULT_DIR / 'comparison_adapter_minus_base.csv', index=False, encoding='utf-8-sig')
    display(comparison.round(4))

,model_name,benchmark,direction,samples,bleu,chrf2,repetition_percent,hit_max_tokens_percent,mean_length_ratio,latency_mean_seconds,latency_p95_seconds
0,base,flores_devtest,en-uz,200,1.2478,15.8707,7.5,0.0,0.6520,0.2353,0.4048
1,base,flores_devtest,uz-en,200,2.4271,22.2743,0.0,1.0,1.1653,0.2652,0.4561
2,base,ntrex,en-uz,200,0.9801,15.4115,12.0,0.0,0.6589,0.2188,0.4564
3,base,ntrex,uz-en,200,1.6972,21.9211,0.0,0.5,1.1812,0.2726,0.5834
4,base,tatoeba_latin,en-uz,200,1.1628,13.4519,0.0,0.0,0.8927,0.0682,0.0975
5,base,tatoeba_latin,uz-en,200,3.8229,15.9702,0.0,0.0,0.8897,0.0622,0.1013
6,adapter,flores_devtest,en-uz,200,1.5830,17.7466,0.5,1.0,0.6995,0.5212,0.8416
7,adapter,flores_devtest,uz-en,200,4.8475,26.5388,0.0,0.0,1.0030,0.3793,0.6331
8,adapter,ntrex,en-uz,200,1.0561,16.2427,0.5,1.5,0.7313,0.5668,1.1330
9,adapter,ntrex,uz-en,200,3.2107,25.2369,0.0,0.0,1.0403,0.3426,0.6091


,benchmark,direction,delta_bleu,delta_chrf2,delta_repetition_percent,delta_hit_max_tokens_percent,delta_latency_p95_seconds
0,flores_devtest,en-uz,0.3352,1.8758,-7.0,1.0,0.4369
1,flores_devtest,uz-en,2.4204,4.2645,0.0,-1.0,0.1770
2,ntrex,en-uz,0.0760,0.8312,-11.5,1.5,0.6766
3,ntrex,uz-en,1.5136,3.3158,0.0,-0.5,0.0257
4,tatoeba_latin,en-uz,-0.2852,0.2040,0.5,0.0,0.0935
5,tatoeba_latin,uz-en,0.7639,4.3725,0.0,0.0,0.0417


In [8]:
manifest = {
    'base_model': BASE_MODEL,
    'adapter_path': str(ADAPTER_PATH),
    'max_pairs_per_direction': MAX_PAIRS_PER_DIRECTION,
    'decode': {
        'num_beams': 1,
        'repetition_penalty': 1.15,
        'no_repeat_ngram_size': 3,
        'max_new_tokens': MAX_NEW_TOKENS,
    },
    'datasets': {
        'flores200': {'source': FLORES_URL, 'license': 'CC-BY-SA-4.0', 'usage': 'evaluation_only'},
        'ntrex128': {'source': 'davidstap/NTREX', 'license': 'CC-BY-SA-4.0', 'usage': 'evaluation_only'},
        'tatoeba': {'source': str(tatoeba_path), 'license': 'CC-BY-2.0', 'usage': 'diagnostic'},
    },
}
(RESULT_DIR / 'evaluation_manifest.json').write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved evaluation artifacts to:', RESULT_DIR)

Saved evaluation artifacts to: D:\dev\projects\fourlang_translation\results\trusted_benchmark_eval
